<!-- # NIST SP 800-90B による min-entropy の評価 -->

# Min-entropy assessment with NIST SP 800-90B

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- 量子乱数生成器（QRNG）の raw data の 1 サンプルあたりの **min-entropy** を、NIST SP 800-90B の non-IID track の方法で推定します。推定には NIST が配布している公式ツール `ea_non_iid` を使います。 -->

We estimate the **min-entropy** per sample of the raw data of a quantum random number generator (QRNG) using the non-IID track of NIST SP 800-90B. For the estimation we use `ea_non_iid`, the official tool distributed by NIST.

> M. S. Turan, E. Barker, J. Kelsey, K. A. McKay, M. L. Baish, and M. Boyle, "Recommendation for the Entropy Sources Used for Random Bit Generation," NIST Special Publication 800-90B (2018).

<!--
1. 公式ツール `ea_non_iid` の使い方を確認する（1.）。ツールのダウンロードとビルドは `README.md` の手順で行う
2. raw data を用意する。Qiskit で C1 回路の QRNG を $10^6$ 回シミュレーションするか、保存した raw data を読み込む（3.）
3. `ea_non_iid` で 10 種類の推定値と、その最小値 $H_{\text{original}}$ を求める（4.）
4. 初期エントロピー推定 $H_I$ を決めて保存する（5.）
-->

1. Check how to use the official tool `ea_non_iid` (1.). Download and build the tool by following `README.md`
2. Prepare the raw data: either simulate the C1-circuit QRNG $10^6$ times with Qiskit, or load saved raw data (3.)
3. Use `ea_non_iid` to obtain the 10 estimates and their minimum $H_{\text{original}}$ (4.)
4. Determine the initial entropy estimate $H_I$ and save it (5.)

<!-- ここで得た $H_I$ は、Toeplitz・Trevisan のノートブックの `H_MIN_OVERRIDE`（抽出に使う 1 ビットあたりの min-entropy $h$）に入れて使います。 -->

The $H_I$ obtained here is used as `H_MIN_OVERRIDE` (the min-entropy per bit $h$ used for extraction) in the Toeplitz and Trevisan notebooks.

---

<!-- ## 手法 -->

## Methods

### A. min-entropy

<!-- サンプルが値 $x_1, \ldots, x_k$ を確率 $p_1, \ldots, p_k$ でとるとき、1 サンプルあたりの min-entropy は -->

When a sample takes the values $x_1, \ldots, x_k$ with probabilities $p_1, \ldots, p_k$, the min-entropy per sample is

$$
H = -\log_2 \max_i p_i
$$

<!-- です。最も出やすい値を 1 回で当てる確率が $2^{-H}$ であることを表します。QRNG では、1 回の測定結果（C1 回路なら 1 ビット）が 1 サンプルです。 -->

This means that the probability of guessing the most likely value in one try is $2^{-H}$. In a QRNG, one measurement outcome (one bit for the C1 circuit) is one sample.

<!-- ### B. non-IID track の 10 種類の推定法（SP 800-90B の 6.3 節） -->

### B. The 10 estimators of the non-IID track (Section 6.3 of SP 800-90B)

<!-- $L$ 個のサンプル $s_1, \ldots, s_L$ から次の 10 種類の推定値を計算し、その**最小値**を $H_{\text{original}}$ とします。どの推定法も、統計量の 99% 信頼区間のうち min-entropy が小さくなる側の端を使います。表の中の $U(\hat{p})$ は、割合 $\hat{p}$ の 99% 信頼区間の上限です。 -->

From $L$ samples $s_1, \ldots, s_L$ we compute the following 10 estimates and take their **minimum** as $H_{\text{original}}$. Every estimator uses the end of the 99% confidence interval of its statistic that gives the smaller min-entropy. In the table, $U(\hat{p})$ is the upper bound of the 99% confidence interval of the proportion $\hat{p}$.

$$
U(\hat{p}) = \min\left(1,\ \hat{p} + 2.576\sqrt{\frac{\hat{p}(1-\hat{p})}{L-1}}\right)
$$

<!--
| 節 | 推定法 | 推定値の式 | 簡単な説明 |
|---|---|---|---|
| 6.3.1 | Most Common Value (MCV) | $-\log_2 U(\hat{p})$ | $\hat{p}$ は最も多い値の割合 |
| 6.3.2 | Collision（2 値のみ） | $-\log_2 p$ | 同じ値が再び出るまでのサンプル数 $t$ の平均の下限 $\bar{X}' = \bar{X} - 2.576\,\hat{\sigma}/\sqrt{v}$ から、$\bar{X}' = p q^{-2}\bigl(1 + \tfrac{1}{2}(p^{-1} - q^{-1})\bigr)F(q) - p q^{-1}\tfrac{1}{2}(p^{-1} - q^{-1})$（$q = 1 - p$、$F(1/z) = \Gamma(3, z) z^{-3} e^{z}$）を満たす $p$ を求める。偏りがあるほど早く同じ値が出る |
| 6.3.3 | Markov（2 値のみ） | $\min\bigl(-\log_2(\hat{p}_{\max})/128,\ 1\bigr)$ | 0 と 1 の割合 $P_0, P_1$ と遷移確率 $P_{a,b}$（$a$ の次に $b$）から、長さ 128 の 6 つの列 $00\cdots0$, $0101\cdots01$, $011\cdots1$, $100\cdots0$, $1010\cdots10$, $11\cdots1$ の確率（たとえば $00\cdots0$ は $P_0 P_{0,0}^{127}$）を計算し、その最大値を $\hat{p}_{\max}$ とする |
| 6.3.4 | Compression（2 値のみ） | $-\log_2(p)/6$ | 6 ビットのブロックに分け、最初の 1000 ブロックを辞書にして、各ブロックが前回出てから何ブロック後に出たか $D_i$ を数える（Maurer の統計量）。$\log_2 D_i$ の平均の下限 $\bar{X}'$ から、$\bar{X}' = G(p) + (2^6 - 1)G(q)$（$q = (1-p)/(2^6-1)$）を満たす $p$ を求める。圧縮しやすいほど値が小さい |
| 6.3.5 | t-Tuple | $-\log_2 U\bigl(\max_{i \le t} P_{\max}[i]\bigr)$ | 長さ $i$ の部分列のうち最も多いものの回数 $Q[i]$ から $P_{\max}[i] = \bigl(Q[i]/(L-i+1)\bigr)^{1/i}$。$t$ は $Q[t] \ge 35$ となる最大の長さ |
| 6.3.6 | Longest Repeated Substring (LRS) | $-\log_2 U\bigl(\max_{u \le W \le v} P_W^{1/W}\bigr)$ | 長さ $W$ の部分列の衝突確率 $P_W = \sum_i \binom{C_i}{2} \big/ \binom{L-W+1}{2}$（$C_i$ は $i$ 番目の部分列の回数）。$u$ は最も多い部分列が 35 回未満になる最小の長さ、$v$ は繰り返しが現れる最大の長さ |
| 6.3.7 | MultiMCW Prediction | 予測の式（下） | 直前の 63, 255, 1023, 4095 サンプルの窓で最も多い値を次の値と予測する |
| 6.3.8 | Lag Prediction | 予測の式（下） | $d$ サンプル前（$d = 1, \ldots, 128$）と同じ値を次の値と予測する |
| 6.3.9 | MultiMMC Prediction | 予測の式（下） | 直前の $d$ 個（$d = 1, \ldots, 16$）の値の後に続いた回数が最も多い値を次の値と予測する |
| 6.3.10 | LZ78Y Prediction | 予測の式（下） | 長さ 16 までの部分列の辞書（最大 65 536 項目）で、直前の部分列の後に最も多く続いた値を次の値と予測する |
-->

| Section | Estimator | Formula for the estimate | Brief description |
|---|---|---|---|
| 6.3.1 | Most Common Value (MCV) | $-\log_2 U(\hat{p})$ | $\hat{p}$ is the proportion of the most common value |
| 6.3.2 | Collision (binary only) | $-\log_2 p$ | From the lower bound $\bar{X}' = \bar{X} - 2.576\,\hat{\sigma}/\sqrt{v}$ of the mean number of samples $t$ until a value repeats, find the $p$ that satisfies $\bar{X}' = p q^{-2}\bigl(1 + \tfrac{1}{2}(p^{-1} - q^{-1})\bigr)F(q) - p q^{-1}\tfrac{1}{2}(p^{-1} - q^{-1})$ ($q = 1 - p$, $F(1/z) = \Gamma(3, z) z^{-3} e^{z}$). The more biased the data, the sooner a value repeats |
| 6.3.3 | Markov (binary only) | $\min\bigl(-\log_2(\hat{p}_{\max})/128,\ 1\bigr)$ | From the proportions $P_0, P_1$ of 0 and 1 and the transition probabilities $P_{a,b}$ ($b$ following $a$), compute the probabilities of the six length-128 sequences $00\cdots0$, $0101\cdots01$, $011\cdots1$, $100\cdots0$, $1010\cdots10$, $11\cdots1$ (for example, $P_0 P_{0,0}^{127}$ for $00\cdots0$), and let $\hat{p}_{\max}$ be the largest |
| 6.3.4 | Compression (binary only) | $-\log_2(p)/6$ | Split the data into 6-bit blocks, use the first 1000 blocks as a dictionary, and count $D_i$, the number of blocks since each block last appeared (Maurer's statistic). From the lower bound $\bar{X}'$ of the mean of $\log_2 D_i$, find the $p$ that satisfies $\bar{X}' = G(p) + (2^6 - 1)G(q)$ ($q = (1-p)/(2^6-1)$). The more compressible the data, the smaller the value |
| 6.3.5 | t-Tuple | $-\log_2 U\bigl(\max_{i \le t} P_{\max}[i]\bigr)$ | From $Q[i]$, the count of the most common length-$i$ subsequence, $P_{\max}[i] = \bigl(Q[i]/(L-i+1)\bigr)^{1/i}$. $t$ is the largest length with $Q[t] \ge 35$ |
| 6.3.6 | Longest Repeated Substring (LRS) | $-\log_2 U\bigl(\max_{u \le W \le v} P_W^{1/W}\bigr)$ | Collision probability of length-$W$ subsequences $P_W = \sum_i \binom{C_i}{2} \big/ \binom{L-W+1}{2}$ ($C_i$ is the count of the $i$-th subsequence). $u$ is the smallest length at which the most common subsequence occurs fewer than 35 times, and $v$ is the largest length at which a repetition occurs |
| 6.3.7 | MultiMCW Prediction | Prediction formula (below) | Predicts the next value as the most common value in windows of the previous 63, 255, 1023, and 4095 samples |
| 6.3.8 | Lag Prediction | Prediction formula (below) | Predicts the next value to be the same as the value $d$ samples earlier ($d = 1, \ldots, 128$) |
| 6.3.9 | MultiMMC Prediction | Prediction formula (below) | Predicts the next value as the value that has most often followed the previous $d$ values ($d = 1, \ldots, 16$) |
| 6.3.10 | LZ78Y Prediction | Prediction formula (below) | Uses a dictionary of subsequences of length up to 16 (at most 65,536 entries) and predicts the next value as the value that has most often followed the preceding subsequence |

<!-- Compression の $G(z)$ は、ブロック数を $L' = \lfloor L/6 \rfloor$、辞書の大きさを $d = 1000$、$v = L' - d$ として $G(z) = \frac{1}{v}\sum_{t=d+1}^{L'}\sum_{u=1}^{t}\log_2(u)\,F(z,t,u)$、$F(z,t,u) = z^2(1-z)^{u-1}$（$u < t$）、$z(1-z)^{t-1}$（$u = t$）です。 -->

In Compression, with $L' = \lfloor L/6 \rfloor$ blocks, a dictionary of size $d = 1000$, and $v = L' - d$, $G(z) = \frac{1}{v}\sum_{t=d+1}^{L'}\sum_{u=1}^{t}\log_2(u)\,F(z,t,u)$, where $F(z,t,u) = z^2(1-z)^{u-1}$ ($u < t$) and $z(1-z)^{t-1}$ ($u = t$).

<!-- **予測にもとづく推定（6.3.7〜6.3.10）の式**：どの予測器も、複数の副予測器のうちそれまでに最も多く当てたものを使って次の値を予測します。$N$ 回の予測のうち $C$ 回当たったとき、 -->

**Formula for the prediction-based estimates (6.3.7–6.3.10)**: each predictor predicts the next value using whichever of its sub-predictors has been correct most often so far. When $C$ of $N$ predictions are correct,

$$
H = -\log_2 \max\left(P'_{\text{global}},\ P_{\text{local}},\ \frac{1}{k}\right), \qquad
P'_{\text{global}} = \begin{cases} 1 - 0.01^{1/N} & (C = 0) \\ \min\left(1,\ P_{\text{global}} + 2.576\sqrt{\dfrac{P_{\text{global}}(1-P_{\text{global}})}{N-1}}\right) & (C > 0) \end{cases}
$$

<!-- です。$P_{\text{global}} = C/N$、$k$ はとりうる値の数（ビット列なら 2）です。$P_{\text{local}}$ は、連続して当たった最長の回数に 1 を足した $r$ から、 -->

where $P_{\text{global}} = C/N$ and $k$ is the number of possible values (2 for a bit string). $P_{\text{local}}$ is the value that satisfies the following, with $r$ equal to the longest run of correct predictions plus 1:

$$
0.99 = \frac{1 - P_{\text{local}}\, x}{(r + 1 - r x)\, q} \cdot \frac{1}{x^{N+1}}, \qquad q = 1 - P_{\text{local}}
$$

<!-- を満たす値です（$x$ は $x_0 = 1$, $x_j = 1 + q P_{\text{local}}^{r} x_{j-1}^{r+1}$ を 10 回繰り返した $x_{10}$）。相関があると予測が当たりやすくなるので、MCV 推定では見落とす相関を捉えられます。 -->

($x$ is $x_{10}$, obtained by iterating $x_0 = 1$, $x_j = 1 + q P_{\text{local}}^{r} x_{j-1}^{r+1}$ ten times). Correlations make predictions more likely to succeed, so these estimators capture correlations that the MCV estimate misses.

<!-- ### C. 初期エントロピー推定（3.1.3 節） -->

### C. Initial entropy estimate (Section 3.1.3)

<!-- 2 値のサンプルでは、データからの推定値 $H_{\text{original}}$ と、ノイズ源の物理的な解析にもとづく提出者の推定値 $H_{\text{submitter}}$ の小さい方を使います。 -->

For binary samples, we use the smaller of the estimate from the data $H_{\text{original}}$ and the submitter's estimate $H_{\text{submitter}}$, which is based on a physical analysis of the noise source.

$$
H_I = \min(H_{\text{original}},\ H_{\text{submitter}})
$$

<!-- ### 但し書き -->

### Caveats

<!--
- **データの量**：ノイズ源から直接得た連続する $10^6$ サンプル以上が必要です（3.1.1 節）。
- **再起動テスト（3.1.4 節）はこのノートブックでは行いません**。エントロピー源を 1000 回再起動して 1000 サンプルずつ集めた行列の行・列をつなげた系列の推定値 $H_r$, $H_c$ が $\min(H_r, H_c) \ge H_I/2$ を満たした場合に、$\min(H_r, H_c, H_I)$ が最終的な評価値になります。量子コンピュータでは「再起動」（ジョブごと、キャリブレーションごとなど）を自分で決める必要があります。公式ツールでは `make restart` で作る `ea_restart` を使います。
- **IID track は使いません**：MCV 推定だけを使う IID track は、5 章の検定で IID が否定されない場合にだけ使えます。読み出し誤差やクロストークで相関が生じうるので、ここでは non-IID track を使います。
- **理想的な乱数でも 1 にはなりません**：信頼区間の端と 10 種類の最小値を使うので、公平で独立なビット列でも $H_{\text{original}}$ は 1 より小さくなります（$10^6$ ビットでは 0.8〜0.9 程度で、多くの場合 Compression 推定が最小値になります）。
- **2 値でないサンプル**：C2 回路の 5 ビットの結果を 1 サンプルとする場合などは、ビットに分けた系列の推定値 $H_{\text{bitstring}}$ も使い、$H_I = \min(H_{\text{original}},\ n H_{\text{bitstring}},\ H_{\text{submitter}})$（$n$ は 1 サンプルのビット数）とします。
-->

- **Amount of data**: At least $10^6$ consecutive samples obtained directly from the noise source are required (Section 3.1.1).
- **This notebook does not perform the restart tests (Section 3.1.4)**. The entropy source is restarted 1000 times and 1000 samples are collected each time; the rows and columns of the resulting matrix are concatenated into sequences with estimates $H_r$ and $H_c$. If $\min(H_r, H_c) \ge H_I/2$, then $\min(H_r, H_c, H_I)$ becomes the final assessed value. On a quantum computer you need to decide for yourself what a "restart" is (per job, per calibration, and so on). The official tool provides `ea_restart`, built with `make restart`.
- **The IID track is not used**: The IID track, which uses only the MCV estimate, may be used only when the tests in Chapter 5 do not reject IID. Readout errors and crosstalk can introduce correlations, so we use the non-IID track here.
- **Even ideal random numbers do not give 1**: Because the end of a confidence interval and the minimum of 10 estimates are used, $H_{\text{original}}$ is smaller than 1 even for a fair, independent bit string (about 0.8–0.9 for $10^6$ bits; the Compression estimate is often the minimum).
- **Non-binary samples**: When, for example, the 5-bit outcome of the C2 circuit is treated as one sample, the estimate $H_{\text{bitstring}}$ of the sequence split into bits is also used, and $H_I = \min(H_{\text{original}},\ n H_{\text{bitstring}},\ H_{\text{submitter}})$ ($n$ is the number of bits per sample).

---

<!-- ## 使い方 -->

## How to use this notebook

<!-- まず `README.md` の手順で Python の環境を作り、このノートブックと同じフォルダに公式ツールをビルドしてから、上のセルから順に実行してください。別の場所に置いた場合は `EA_NON_IID` を書き換えてください。条件は、次の変数を書き換えて選びます。 -->

First set up the Python environment and build the official tool in the same folder as this notebook by following `README.md`, then run the cells in order from the top. If you placed the tool elsewhere, edit `EA_NON_IID`. The conditions are selected by editing the following variables.

<!--
| 変数 | 意味 | 既定値 |
|---|---|---|
| `EA_NON_IID` | ビルドした `ea_non_iid` のパス | `SP800-90B_EntropyAssessment/cpp/ea_non_iid`（このノートブックからの相対パス） |
| `SAVE_RESULTS` | 結果を保存するか（`True` なら `results/sp800_90b/<日時>/` に保存） | `False` |
| `INPUT_PATH` | raw data のファイル（`None` ならシミュレーションを使う） | `None` |
| `GATE` | C1 回路のゲート（`"H"`, `"Rx"`, `"Ry"`） | `"H"` |
| `SHOTS` | 測定回数（= サンプル数） | `1_000_000` |
| `READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1` | 読み出し誤差 $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.03`, `0.03` |
| `SIM_SEED` | シミュレータの乱数シード（`None` なら毎回違う結果） | `1` |
| `H_SUBMITTER` | 提出者の推定値 $H_{\text{submitter}}$（`None` なら使わない） | `None` |
-->

| Variable | Meaning | Default |
|---|---|---|
| `EA_NON_IID` | Path to the built `ea_non_iid` | `SP800-90B_EntropyAssessment/cpp/ea_non_iid` (relative to this notebook) |
| `SAVE_RESULTS` | Whether to save the results (`True` saves them in `results/sp800_90b/<date_time>/`) | `False` |
| `INPUT_PATH` | Raw-data file (`None` means use the simulation) | `None` |
| `GATE` | Gate of the C1 circuit (`"H"`, `"Rx"`, `"Ry"`) | `"H"` |
| `SHOTS` | Number of measurements (= number of samples) | `1_000_000` |
| `READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1` | Readout errors $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.03`, `0.03` |
| `SIM_SEED` | Random seed of the simulator (`None` gives different results each time) | `1` |
| `H_SUBMITTER` | Submitter's estimate $H_{\text{submitter}}$ (`None` means not used) | `None` |

<!-- `SAVE_RESULTS = True` のときは、2. の準備のセルを実行するたびに新しい日時のフォルダに保存するので、前の結果を上書きしません。 -->

When `SAVE_RESULTS = True`, each run of the setup cell in 2. saves into a new date-and-time folder, so earlier results are never overwritten.

<!-- ### 保存されるファイル -->

### Saved files

<!-- 2. の準備のセルで `SAVE_RESULTS = True` にすると、そのセルを実行した日時の名前のフォルダ `results/sp800_90b/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作り、次のファイルを保存します。`False`（既定）では、ツールに渡すファイルを一時フォルダに書くだけで、何も保存しません。 -->

If you set `SAVE_RESULTS = True` in the setup cell in 2., a folder named after the date and time that cell was run, `results/sp800_90b/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created and the following files are saved in it. With `False` (the default), the file for the tool is only written to a temporary folder and nothing is saved.

<!--
```
results/sp800_90b/
└── 2026-11-10_153012/                RUN_DIR（2. のセルを実行した日時）
    ├── c1_H_sim.bin                     ツールに渡した raw data
    ├── c1_H_sim_ea_non_iid.txt          ツールの出力の全文
    └── c1_H_sim_estimates.csv           推定結果
```
-->

```
results/sp800_90b/
└── 2026-11-10_153012/                RUN_DIR (date and time the cell in 2. was run)
    ├── c1_H_sim.bin                     raw data passed to the tool
    ├── c1_H_sim_ea_non_iid.txt          full output of the tool
    └── c1_H_sim_estimates.csv           estimates
```

<!-- ファイル名の先頭の `<data_label>` は、シミュレーションなら `c1_<GATE>_sim`（例：`c1_H_sim`）、`INPUT_PATH` を使う場合はそのファイル名から拡張子を除いたもの（例：C1 のノートブックで保存した `c1_H_ideal_trial001`）です。 -->

`<data_label>` at the start of each file name is `c1_<GATE>_sim` (e.g., `c1_H_sim`) for the simulation, or the file name of `INPUT_PATH` without its extension (e.g., `c1_H_ideal_trial001` saved by the C1 notebook) when `INPUT_PATH` is used.

<!--
| ファイル | 形式 | 中身 | 書き出すセル |
|---|---|---|---|
| `<data_label>.bin` | 2 進ファイル。1 サンプル = 1 バイト（値は `0x00` か `0x01`）で、文字の `'0'`, `'1'` ではない | `ea_non_iid` に渡した raw data（$10^6$ サンプルなら 1,000,000 バイト） | 4.2 |
| `<data_label>_ea_non_iid.txt` | テキスト | `ea_non_iid` の出力の全文。入力ファイルの SHA-256、10 種類の推定値、最後の行に `H_original: ...` | 4.2 |
| `<data_label>_estimates.csv` | CSV（ヘッダー 1 行 + データ 1 行） | 推定の結果。列は下の表 | 5. |
-->

| File | Format | Contents | Written by cell |
|---|---|---|---|
| `<data_label>.bin` | Binary file. 1 sample = 1 byte (value `0x00` or `0x01`), not the characters `'0'`, `'1'` | The raw data passed to `ea_non_iid` (1,000,000 bytes for $10^6$ samples) | 4.2 |
| `<data_label>_ea_non_iid.txt` | Text | Full output of `ea_non_iid`: the SHA-256 of the input file, the 10 estimates, and `H_original: ...` on the last line | 4.2 |
| `<data_label>_estimates.csv` | CSV (1 header row + 1 data row) | The estimation results. Columns are listed in the table below | 5. |

<!-- `<data_label>_estimates.csv` の列は次のとおりです。 -->

The columns of `<data_label>_estimates.csv` are as follows.

<!--
| 列 | 意味 |
|---|---|
| `data` | `<data_label>` |
| `sha256` | 評価した `.bin` ファイルの SHA-256（ツールの出力の値と同じ） |
| `samples` | サンプル数 |
| `tool_commit` | 使った公式ツールの git のコミット |
| `Most Common Value` 〜 `LZ78Y Prediction Test` | 10 種類の推定値（1 サンプルあたりの min-entropy）。列の名前はツールの出力の名前のまま |
| `H_original` | 10 種類の推定値の最小値 $H_{\text{original}}$ |
| `H_submitter` | 提出者の推定値 $H_{\text{submitter}}$（`H_SUBMITTER = None` なら空欄） |
| `H_I` | 初期エントロピー推定 $H_I = \min(H_{\text{original}}, H_{\text{submitter}})$ |
-->

| Column | Meaning |
|---|---|
| `data` | `<data_label>` |
| `sha256` | SHA-256 of the assessed `.bin` file (the same value as in the tool's output) |
| `samples` | Number of samples |
| `tool_commit` | Git commit of the official tool used |
| `Most Common Value` to `LZ78Y Prediction Test` | The 10 estimates (min-entropy per sample). Column names are the names in the tool's output |
| `H_original` | Minimum of the 10 estimates, $H_{\text{original}}$ |
| `H_submitter` | Submitter's estimate $H_{\text{submitter}}$ (empty if `H_SUBMITTER = None`) |
| `H_I` | Initial entropy estimate $H_I = \min(H_{\text{original}}, H_{\text{submitter}})$ |

<!-- - ファイルは上書きしません。同じ `RUN_DIR` のまま 4.2 や 5. のセルをもう一度実行するとエラー（`FileExistsError`）になるので、2. の準備のセルを実行し直して新しいフォルダを作ってください。 -->

- Files are never overwritten. If you run the cell in 4.2 or 5. again with the same `RUN_DIR`, an error (`FileExistsError`) is raised, so rerun the setup cell in 2. to create a new folder.

<!-- ### 関数の家系図 -->

### Function call tree

<!-- このノートブックの関数の呼び出し関係です。上の（字下げの浅い）関数が、そのすぐ下に枝で続く関数を内部で呼び出します。 -->

This shows how the functions in this notebook call each other. A function higher up (less indented) internally calls the functions on the branches directly below it.

<!--
- `[3.]` のような右の番号は、**その関数が定義されている章**です。
- `← 4.2 のセルから実行` は、その章のコードセルから直接呼び出していることを示します。
- `（↑ 上で展開済み）` は、同じ関数の中身をすでに上の図で展開していることを示します。
-->

- The number on the right, such as `[3.]`, is **the section in which the function is defined**.
- `← run from the cell in 4.2` means the function is called directly from the code cell of that section.
- `(↑ expanded above)` means the contents of the same function have already been expanded earlier in the tree.

<!-- **raw data の準備** -->

**Preparing the raw data**

<!--
```
simulate_c1_bits   [3.]  ← 3. のセルから実行
└── build_c1       [3.]

build_noise_model  [3.]  ← 3. のセルから実行
load_bits          [3.]  ← 3. のセルから実行
```
-->

```
simulate_c1_bits   [3.]  ← run from the cell in 3.
└── build_c1       [3.]

build_noise_model  [3.]  ← run from the cell in 3.
load_bits          [3.]  ← run from the cell in 3.
```

<!-- **ツールの確認と min-entropy の推定** -->

**Checking the tool and estimating the min-entropy**

<!--
```
tool_version     [2.]  ← 2.・5. のセルから実行
write_90b_input  [4.1]  ← 4.2 のセルから実行
run_ea_non_iid   [4.1]  ← 4.2 のセルから実行
show_estimates   [4.1]  ← 4.2 のセルから実行
```
-->

```
tool_version     [2.]  ← run from the cells in 2. and 5.
write_90b_input  [4.1]  ← run from the cell in 4.2
run_ea_non_iid   [4.1]  ← run from the cell in 4.2
show_estimates   [4.1]  ← run from the cell in 4.2
```

<!-- **章ごとの定義の一覧** -->

**Functions defined in each section**

<!--
| 章 | 章の題 | 定義されている関数 |
|---|---|---|
| 2. | 準備 | `tool_version` |
| 3. | raw data を用意する | `build_c1`、`build_noise_model`、`simulate_c1_bits`、`load_bits` |
| 4.1 | ツールを使う関数 | `write_90b_input`、`run_ea_non_iid`、`show_estimates` |
-->

| Section | Section title | Functions defined |
|---|---|---|
| 2. | Setup | `tool_version` |
| 3. | Preparing the raw data | `build_c1`, `build_noise_model`, `simulate_c1_bits`, `load_bits` |
| 4.1 | Functions for using the tool | `write_90b_input`, `run_ea_non_iid`, `show_estimates` |

<!-- ## 1. 公式ツール `ea_non_iid` -->

## 1. The official tool `ea_non_iid`

<!--
- リポジトリ：<https://github.com/usnistgov/SP800-90B_EntropyAssessment>
- SP 800-90B の本文：<https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-90B.pdf>
-->

- Repository: <https://github.com/usnistgov/SP800-90B_EntropyAssessment>
- Text of SP 800-90B: <https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-90B.pdf>

<!-- `ea_non_iid` のダウンロードとビルドは、リポジトリの `README.md` の手順で行います。その手順どおりにすると、`ea_non_iid` はこのノートブックと同じフォルダの `SP800-90B_EntropyAssessment/cpp/ea_non_iid` にビルドされます（コミット `87c104d0ed4cbc96103e7b8b38d6f2c7e0a6b289` で確認しました）。このパスが `EA_NON_IID` です。 -->

Download and build `ea_non_iid` by following `README.md` in the repository. If you follow those steps, `ea_non_iid` is built at `SP800-90B_EntropyAssessment/cpp/ea_non_iid` in the same folder as this notebook (verified with commit `87c104d0ed4cbc96103e7b8b38d6f2c7e0a6b289`). This path is `EA_NON_IID`.

<!-- **`ea_non_iid` の使い方** -->

**How to use `ea_non_iid`**

```bash
./ea_non_iid -i -v <file_name> <bits_per_symbol>
```

<!--
| 引数 | 意味 |
|---|---|
| `<file_name>` | 評価するファイル。**1 サンプルを 1 バイト**に入れた 2 進ファイル |
| `<bits_per_symbol>` | 1 サンプルのビット数（1〜8）。C1 回路のビット列なら `1` |
| `-i` | 初期エントロピー推定（3.1.3 節）。IID の宣言ではありません |
| `-v` | 各推定法の途中の値も表示する |
-->

| Argument | Meaning |
|---|---|
| `<file_name>` | File to assess. A binary file with **one sample per byte** |
| `<bits_per_symbol>` | Number of bits per sample (1–8). `1` for the bit string of the C1 circuit |
| `-i` | Initial entropy estimate (Section 3.1.3). This is not a claim of IID |
| `-v` | Also prints the intermediate values of each estimator |

<!-- 出力の最後の行 `H_original: ...` が 10 種類の推定値の最小値です。 -->

The last line of the output, `H_original: ...`, is the minimum of the 10 estimates.

<!-- > **入力ファイルの形式**：ビット列を `"0101..."` という文字のまま保存すると、1 バイトの値は `0x30` と `0x31` になり、意図した 0 と 1 になりません。4. の `write_90b_input` は 0 と 1 をそのまま 1 バイトずつ書きます。 -->

> **Input file format**: If a bit string is saved as the characters `"0101..."`, the byte values become `0x30` and `0x31`, not the intended 0 and 1. `write_90b_input` in 4. writes 0 and 1 as they are, one per byte.

<!-- ## 2. 準備 -->

## 2. Setup

<!-- ライブラリを読み込み、公式ツールの場所 `EA_NON_IID` と、結果を保存するかどうか `SAVE_RESULTS` を指定します。ツールが見つからない場合は、ツールを使うセルは実行せず、そのことを表示します。 -->

We load the libraries and specify the location of the official tool, `EA_NON_IID`, and whether to save the results, `SAVE_RESULTS`. If the tool is not found, the cells that use it are skipped and a message says so.

<!-- 結果の保存：`SAVE_RESULTS` で、ツールに渡したファイルと推定結果を保存するかどうかを選びます。`True` にすると、このセルを実行した日時の名前のフォルダ（例：`results/sp800_90b/2026-11-10_153012/`）を作り、その中に保存します。`False`（既定）では、ツールに渡すファイルを一時フォルダに書き、何も残しません。 -->

Saving the results: `SAVE_RESULTS` selects whether the file passed to the tool and the estimates are saved. With `True`, a folder named after the date and time this cell was run (e.g., `results/sp800_90b/2026-11-10_153012/`) is created and the files are saved in it. With `False` (the default), the file for the tool is written to a temporary folder and nothing is kept.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `tool_version` | 評価に使ったツールの版を記録するために、ツールのフォルダの git のコミットを取得することを目的とした関数です。 | ツールのフォルダの git のコミット（str）。取得できなければ `"unknown"` | `tool_version(EA_NON_IID)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `tool_version` | Gets the git commit of the tool's folder, to record which version of the tool was used for the assessment. | Git commit of the tool's folder (str); `"unknown"` if it cannot be obtained | `tool_version(EA_NON_IID)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>tool_version</code></td><td><code>path</code></td><td><code>ea_non_iid</code> のパス</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>tool_version</code></td><td><code>path</code></td><td>Path to <code>ea_non_iid</code></td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
import csv
import hashlib
import math
import re
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error, ReadoutError
from IPython.display import Markdown, display

# Location of ea_non_iid built by following README.md (extracted into the same folder as this notebook)
EA_NON_IID = Path("SP800-90B_EntropyAssessment/cpp/ea_non_iid").resolve()

# True: save the files of this run in a new folder results/sp800_90b/<date_time>/; False: save nothing
SAVE_RESULTS = False
RESULTS_ROOT = Path("results/sp800_90b")
RUN_DIR = RESULTS_ROOT / datetime.now().strftime("%Y-%m-%d_%H%M%S") if SAVE_RESULTS else None
print("save to    :", RUN_DIR if RUN_DIR is not None else "nothing is saved (SAVE_RESULTS = False)")
# Folder for the file passed to the tool: RUN_DIR when saving, otherwise a temporary folder (nothing is kept)
DATA_DIR = RUN_DIR if RUN_DIR is not None else Path(tempfile.mkdtemp(prefix="sp800_90b_"))
DATA_DIR.mkdir(parents=True, exist_ok=True)


def tool_version(path):
    try:
        out = subprocess.run(["git", "-C", str(Path(path).parent), "rev-parse", "HEAD"],
                             capture_output=True, text=True, check=True)
        return out.stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return "unknown"


TOOL_AVAILABLE = EA_NON_IID.is_file()
if TOOL_AVAILABLE:
    print("ea_non_iid :", EA_NON_IID)
    print("commit     :", tool_version(EA_NON_IID))
else:
    print(f"ea_non_iid not found: {EA_NON_IID}")
    print("Build it in the same folder as this notebook by following README.md, or edit EA_NON_IID. Cells that use the tool will be skipped.")

<!-- ## 3. raw data を用意する -->

## 3. Preparing the raw data

<!-- C1 回路（$|0\rangle$ に $H$, $R_X(\pi/2)$, $R_Y(\pi/2)$ のいずれかを作用させて測定する回路）を Qiskit Aer でシミュレーションします。回路はネイティブゲート $R_X$, $R_Y$, $\mathrm{CZ}$ に変換し（`optimization_level=0`）、次のノイズを加えます。 -->

We simulate the C1 circuit (a circuit that applies one of $H$, $R_X(\pi/2)$, $R_Y(\pi/2)$ to $|0\rangle$ and measures) with Qiskit Aer. The circuit is converted to the native gates $R_X$, $R_Y$, $\mathrm{CZ}$ (`optimization_level=0`), and the following noise is added.

<!--
| ノイズ | 値 |
|---|---|
| 1 量子ビットゲートの脱分極誤差 | $\lambda = 0.00198$ |
| 1 量子ビットゲートの熱緩和誤差 | $T_1 = 0.964$ ms, $T_2 = 1.155$ ms, ゲート時間 20 ns |
| 読み出し誤差 | $P(1 \mid 0)$, $P(0 \mid 1)$（`READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1`） |
-->

| Noise | Value |
|---|---|
| Depolarizing error of 1-qubit gates | $\lambda = 0.00198$ |
| Thermal relaxation error of 1-qubit gates | $T_1 = 0.964$ ms, $T_2 = 1.155$ ms, gate time 20 ns |
| Readout error | $P(1 \mid 0)$, $P(0 \mid 1)$ (`READOUT_P1_GIVEN_0`, `READOUT_P0_GIVEN_1`) |

<!-- `INPUT_PATH` を指定すると、シミュレーションの代わりにそのファイルの raw data（0 と 1 の文字だけのテキスト）を使います。 -->

If `INPUT_PATH` is set, the raw data in that file (a text file of only the characters 0 and 1) is used instead of the simulation.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `build_c1` | C1 回路（1 量子ビットに重ね合わせを作るゲートを 1 個作用させて測定する回路）を作ることを目的とした関数です。 | 測定つきの `QuantumCircuit` | `build_c1("H")` |
| `build_noise_model` | 実機を模したシミュレーションのために、ゲートノイズと読み出し誤差を含むノイズモデルを作ることを目的とした関数です。 | `NoiseModel` | `build_noise_model(0.03, 0.03)` |
| `simulate_c1_bits` | C1 回路を Qiskit でシミュレーションし、raw data に相当する測定順のビット列を得ることを目的とした関数です。 | 測定順のビット列（長さ `shots` の `np.uint8` の配列） | `simulate_c1_bits("H", 1000, build_noise_model(0.03, 0.03), seed=1)` |
| `load_bits` | 保存した raw data の 0 と 1 のテキストファイルを読み込み、評価の入力に使える配列に変換することを目的とした関数です。 | ファイルのビット列（`np.uint8` の配列） | `load_bits("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `build_c1` | Builds the C1 circuit (a circuit that applies one superposition-creating gate to one qubit and measures it). | `QuantumCircuit` with measurement | `build_c1("H")` |
| `build_noise_model` | Builds a noise model with gate noise and readout errors, for a simulation that mimics real hardware. | `NoiseModel` | `build_noise_model(0.03, 0.03)` |
| `simulate_c1_bits` | Simulates the C1 circuit with Qiskit and obtains the bit string in measurement order, which corresponds to the raw data. | Bit string in measurement order (`np.uint8` array of length `shots`) | `simulate_c1_bits("H", 1000, build_noise_model(0.03, 0.03), seed=1)` |
| `load_bits` | Reads a saved raw-data text file of 0s and 1s and converts it into an array that can be used as input for the assessment. | Bit string of the file (`np.uint8` array) | `load_bits("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>build_c1</code></td><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td rowspan="3"><code>build_noise_model</code></td><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>gate_noise</code></td><td>ゲートノイズ（脱分極 + 熱緩和）を加えるなら <code>True</code></td><td><code>True</code></td></tr>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>測定回数</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td><code>NoiseModel</code>。<code>None</code> ならノイズなし</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>シミュレータの乱数シード（<code>None</code> なら毎回違う結果）</td><td><code>None</code></td></tr>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td>0 と 1 の文字だけのテキストファイル（空白・改行は無視）</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>build_c1</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td rowspan="3"><code>build_noise_model</code></td><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>gate_noise</code></td><td><code>True</code> to add gate noise (depolarizing + thermal relaxation)</td><td><code>True</code></td></tr>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td><code>NoiseModel</code>. <code>None</code> means no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Random seed of the simulator (<code>None</code> gives different results each time)</td><td><code>None</code></td></tr>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td>Text file of only the characters 0 and 1 (whitespace and newlines are ignored)</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
BASIS_GATES = ["rx", "ry", "cz"]
GATES = ["H", "Rx", "Ry"]
DEPOLARIZING_PARAM = 0.00198
T1, T2, GATE_TIME = 0.964e6, 1.155e6, 20   # ns


def build_c1(gate):
    if gate not in GATES:
        raise ValueError(f"gate must be one of {GATES} (got: {gate!r})")
    qc = QuantumCircuit(1, 1)
    if gate == "H":
        qc.h(0)
    elif gate == "Rx":
        qc.rx(math.pi / 2, 0)
    else:
        qc.ry(math.pi / 2, 0)
    qc.measure(0, 0)
    return qc


def build_noise_model(p1_given_0=0.0, p0_given_1=0.0, gate_noise=True):
    noise_model = NoiseModel(basis_gates=BASIS_GATES)
    if gate_noise:
        error = depolarizing_error(DEPOLARIZING_PARAM, 1).compose(thermal_relaxation_error(T1, T2, GATE_TIME))
        noise_model.add_all_qubit_quantum_error(error, ["rx", "ry"])
    if p1_given_0 or p0_given_1:
        noise_model.add_all_qubit_readout_error(ReadoutError([
            [1 - p1_given_0, p1_given_0],   # true value 0: P(0|0), P(1|0)
            [p0_given_1, 1 - p0_given_1],   # true value 1: P(0|1), P(1|1)
        ]))
    return noise_model


def simulate_c1_bits(gate, shots, noise_model=None, seed=None):
    qc_t = transpile(build_c1(gate), basis_gates=BASIS_GATES, optimization_level=0)
    backend = AerSimulator(noise_model=noise_model, seed_simulator=seed)
    memory = backend.run(qc_t, shots=shots, memory=True).result().get_memory()   # list of "0"/"1" in measurement order
    return (np.frombuffer("".join(memory).encode("ascii"), dtype=np.uint8) - ord("0")).astype(np.uint8)


def load_bits(path):
    text = "".join(Path(path).read_text(encoding="ascii").split())
    bits = np.frombuffer(text.encode("ascii"), dtype=np.uint8) - ord("0")
    if bits.size == 0 or np.any(bits > 1):
        raise ValueError(f"the file contains characters other than 0 and 1, or is empty: {path}")
    return bits.astype(np.uint8)

In [ ]:
INPUT_PATH = None   # e.g., Path("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")
GATE = "H"
SHOTS = 1_000_000
READOUT_P1_GIVEN_0 = 0.03
READOUT_P0_GIVEN_1 = 0.03
SIM_SEED = 1

if INPUT_PATH is None:
    raw_bits = simulate_c1_bits(GATE, SHOTS, build_noise_model(READOUT_P1_GIVEN_0, READOUT_P0_GIVEN_1), seed=SIM_SEED)
    data_label = f"c1_{GATE}_sim"
else:
    raw_bits = load_bits(INPUT_PATH)
    data_label = Path(INPUT_PATH).stem
print(f"{data_label}: {raw_bits.size} samples, P(1) = {raw_bits.mean():.5f}")
if raw_bits.size < 1_000_000:
    print("Note: SP 800-90B requires at least 10^6 samples.")

<!-- ## 4. `ea_non_iid` で min-entropy を推定する -->

## 4. Estimating the min-entropy with `ea_non_iid`

<!-- ### 4.1 ツールを使う関数 -->

### 4.1 Functions for using the tool

<!-- `write_90b_input` で 1 サンプル 1 バイトのファイルを作り、`run_ea_non_iid` で `ea_non_iid -i -v <file> <bits_per_symbol>` を実行して、出力の `... Estimate = 0.995221 / 1 bit(s)` の行と `H_original: ...` の行を読み取ります。ファイルの SHA-256 は、ツールが表示する値と照合できます（評価したファイルの記録）。 -->

`write_90b_input` writes a file with one sample per byte, and `run_ea_non_iid` runs `ea_non_iid -i -v <file> <bits_per_symbol>` and reads the `... Estimate = 0.995221 / 1 bit(s)` lines and the `H_original: ...` line of the output. The SHA-256 of the file can be checked against the value the tool prints (a record of which file was assessed).

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `write_90b_input` | ビット列を `ea_non_iid` が読める形式（1 サンプル 1 バイトの 2 進ファイル）で書き出し、照合用の SHA-256 を求めることを目的とした関数です。 | ファイルの SHA-256（16 進数の文字列） | `write_90b_input(DATA_DIR / "c1.bin", raw_bits)` |
| `run_ea_non_iid` | `ea_non_iid` を実行し、その出力から各推定法の推定値と $H_{\text{original}}$ を読み取ることを目的とした関数です。 | `estimates`（推定法の名前 → 値の辞書）、`H_original`、`sha256`、`stdout` をキーとする辞書 | `run_ea_non_iid(DATA_DIR / "c1.bin")` |
| `show_estimates` | 各推定法の min-entropy の推定値を、最小値が分かる形の表で表示することを目的とした関数です。 | 推定値の表を表示する（最小値を太字）。戻り値は `None` | `show_estimates({"C1": result})` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `write_90b_input` | Writes a bit string in the format `ea_non_iid` reads (a binary file with one sample per byte) and computes its SHA-256 for verification. | SHA-256 of the file (hexadecimal string) | `write_90b_input(DATA_DIR / "c1.bin", raw_bits)` |
| `run_ea_non_iid` | Runs `ea_non_iid` and reads the estimate of each estimator and $H_{\text{original}}$ from its output. | Dictionary with the keys `estimates` (dictionary of estimator name → value), `H_original`, `sha256`, and `stdout` | `run_ea_non_iid(DATA_DIR / "c1.bin")` |
| `show_estimates` | Displays the min-entropy estimate of each estimator in a table that shows which one is the minimum. | Displays the table of estimates (minimum in bold). Returns `None` | `show_estimates({"C1": result})` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>write_90b_input</code></td><td><code>path</code></td><td>保存先。既に存在する場合は上書き</td><td>必須</td></tr>
<tr><td><code>samples</code></td><td>サンプルの配列。各値は 0〜255（ビット列なら 0 と 1）</td><td>必須</td></tr>
<tr><td rowspan="4"><code>run_ea_non_iid</code></td><td><code>path</code></td><td><code>write_90b_input</code> で作ったファイル</td><td>必須</td></tr>
<tr><td><code>bits_per_symbol</code></td><td>1 サンプルのビット数（1〜8）</td><td><code>1</code></td></tr>
<tr><td><code>tool</code></td><td><code>ea_non_iid</code> のパス。<code>None</code> なら <code>EA_NON_IID</code></td><td><code>None</code></td></tr>
<tr><td><code>verbose</code></td><td><code>-v</code> をつけるなら <code>True</code></td><td><code>True</code></td></tr>
<tr><td><code>show_estimates</code></td><td><code>results</code></td><td>列の見出し → <code>run_ea_non_iid</code> の戻り値 の辞書</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>write_90b_input</code></td><td><code>path</code></td><td>Destination. Overwritten if it already exists</td><td>Required</td></tr>
<tr><td><code>samples</code></td><td>Array of samples. Each value is 0–255 (0 and 1 for a bit string)</td><td>Required</td></tr>
<tr><td rowspan="4"><code>run_ea_non_iid</code></td><td><code>path</code></td><td>File created by <code>write_90b_input</code></td><td>Required</td></tr>
<tr><td><code>bits_per_symbol</code></td><td>Number of bits per sample (1–8)</td><td><code>1</code></td></tr>
<tr><td><code>tool</code></td><td>Path to <code>ea_non_iid</code>. <code>None</code> means <code>EA_NON_IID</code></td><td><code>None</code></td></tr>
<tr><td><code>verbose</code></td><td><code>True</code> to add <code>-v</code></td><td><code>True</code></td></tr>
<tr><td><code>show_estimates</code></td><td><code>results</code></td><td>Dictionary of column heading → return value of <code>run_ea_non_iid</code></td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def write_90b_input(path, samples):
    samples = np.asarray(samples).reshape(-1)
    if samples.min() < 0 or samples.max() > 255:
        raise ValueError("samples must be in 0..255")
    data = samples.astype(np.uint8).tobytes()   # 1 sample = 1 byte
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return hashlib.sha256(data).hexdigest()


ESTIMATE_LINE = re.compile(r"^\s*(.+?) Estimate = ([0-9.eE+-]+) / (\d+) bit")
H_ORIGINAL_LINE = re.compile(r"^H_original: ([0-9.eE+-]+)")
SHA_LINE = re.compile(r"SHA-256 hash ([0-9a-f]{64})")


def run_ea_non_iid(path, bits_per_symbol=1, tool=None, verbose=True):
    tool = EA_NON_IID if tool is None else Path(tool)
    command = [str(tool), "-i"] + (["-v"] if verbose else []) + [str(path), str(bits_per_symbol)]
    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"ea_non_iid failed ({completed.returncode}):\n{completed.stdout}\n{completed.stderr}")
    estimates, h_original, sha256 = {}, None, None
    for line in completed.stdout.splitlines():
        if (m := ESTIMATE_LINE.match(line)):
            estimates[m.group(1).strip()] = float(m.group(2))
        elif (m := H_ORIGINAL_LINE.match(line)):
            h_original = float(m.group(1))
        elif (m := SHA_LINE.search(line)):
            sha256 = m.group(1)
    if h_original is None:
        raise RuntimeError("H_original was not found in the output:\n" + completed.stdout)
    return {"estimates": estimates, "H_original": h_original, "sha256": sha256, "stdout": completed.stdout}


def show_estimates(results):
    labels = list(results)
    names = list(dict.fromkeys(name for r in results.values() for name in r["estimates"]))
    lines = ["| estimator | " + " | ".join(labels) + " |", "|---|" + "---|" * len(labels)]
    for name in names:
        cells = []
        for label in labels:
            value = results[label]["estimates"].get(name)
            text = "-" if value is None else f"{value:.6f}"
            if value is not None and value == results[label]["H_original"]:
                text = f"**{text}**"   # minimum of the column
            cells.append(text)
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    lines.append("| **H_original (min)** | " + " | ".join(f"**{results[l]['H_original']:.6f}**" for l in labels) + " |")
    display(Markdown("\n".join(lines)))

<!-- ### 4.2 推定を実行する -->

### 4.2 Running the estimation

<!-- 3. の raw data をファイルにして `ea_non_iid` を実行します（$10^6$ サンプルで数秒〜数十秒かかります）。ファイルは `DATA_DIR`（保存するときは `RUN_DIR`、しないときは一時フォルダ）に書き、ツールの出力の全文を `<data_label>_ea_non_iid.txt` に保存します。予測にもとづく 4 種類の推定値は、ツールの出力では `... Prediction Test Estimate` と表示されます。 -->

We write the raw data from 3. to a file and run `ea_non_iid` (a few seconds to a few tens of seconds for $10^6$ samples). The file is written to `DATA_DIR` (`RUN_DIR` when saving, otherwise a temporary folder), and the full output of the tool is saved as `<data_label>_ea_non_iid.txt`. In the tool's output, the four prediction-based estimates are shown as `... Prediction Test Estimate`.

In [ ]:
input_path = DATA_DIR / f"{data_label}.bin"
log_path = DATA_DIR / f"{data_label}_ea_non_iid.txt"
if RUN_DIR is not None and (input_path.exists() or log_path.exists()):
    raise FileExistsError(f"Destination already exists: {input_path}. Run the setup cell (2.) again to save in a new folder.")
input_sha = write_90b_input(input_path, raw_bits)
print(f"{input_path}: {input_path.stat().st_size} bytes, SHA-256 = {input_sha}")

if TOOL_AVAILABLE:
    result = run_ea_non_iid(input_path)
    log_path.write_text(result["stdout"], encoding="utf-8")   # full output of the tool
    print("Matches the tool's SHA-256:", result["sha256"] == input_sha)
    show_estimates({data_label: result})
else:
    print("ea_non_iid is not available, so this cell is skipped.")

<!-- ## 5. 初期エントロピー推定 $H_I$ を決めて保存する -->

## 5. Determining and saving the initial entropy estimate $H_I$

<!-- 手法 C の $H_I = \min(H_{\text{original}}, H_{\text{submitter}})$ を計算します。`SAVE_RESULTS = True` なら、推定値を `RUN_DIR` の `<data_label>_estimates.csv` に保存します。$H_{\text{submitter}}$ を与えない場合（`H_SUBMITTER = None`）は $H_I = H_{\text{original}}$ です。 -->

We compute $H_I = \min(H_{\text{original}}, H_{\text{submitter}})$ from Method C. If `SAVE_RESULTS = True`, the estimates are saved to `<data_label>_estimates.csv` in `RUN_DIR`. If $H_{\text{submitter}}$ is not given (`H_SUBMITTER = None`), $H_I = H_{\text{original}}$.

In [ ]:
H_SUBMITTER = None   # e.g., a value based on an analysis of the noise source

if TOOL_AVAILABLE:
    h_original = result["H_original"]
    H_I = h_original if H_SUBMITTER is None else min(h_original, H_SUBMITTER)
    print(f"{data_label}: H_original = {h_original:.6f}, H_I = {H_I:.6f}")
    print(f"In the extractor notebooks, set H_MIN_OVERRIDE = {H_I:.6f}.")

    if RUN_DIR is None:
        print("SAVE_RESULTS = False, so the estimates are not saved.")
    else:
        csv_path = RUN_DIR / f"{data_label}_estimates.csv"
        names = list(result["estimates"])
        with csv_path.open("x", newline="", encoding="utf-8") as f:   # existing files are never overwritten
            writer = csv.writer(f)
            writer.writerow(["data", "sha256", "samples", "tool_commit"] + names + ["H_original", "H_submitter", "H_I"])
            writer.writerow([data_label, input_sha, raw_bits.size, tool_version(EA_NON_IID)]
                            + [result["estimates"][name] for name in names]
                            + [h_original, "" if H_SUBMITTER is None else H_SUBMITTER, H_I])
        print("Saved to:", csv_path)
else:
    print("ea_non_iid is not available, so H_I is not computed.")